# Building segmentation under class imbalance — Dar es Salaam

Runs the whole experiment on a Colab GPU. Set the runtime to **T4 GPU** before
starting (Runtime -> Change runtime type -> T4 GPU), or the eight arms will take
many hours instead of roughly one.

Nothing here reimplements the pipeline: every cell calls the same modules that
run locally, so a result produced in Colab and a result produced on a worksta-
tion come from identical code.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "no GPU - switch the runtime to T4"


## 1. Get the code

Replace the URL if you are working from a fork.

In [ ]:
REPO = "https://github.com/WMahendra/dar-building-segmentation.git"

import os, pathlib
if not pathlib.Path("dar-building-segmentation").exists():
    !git clone -q $REPO
%cd dar-building-segmentation
!pip install -q -r requirements.txt


## 2. Download the data

`--dry-run` first: it lists every file and its size without fetching anything,
so you know what the download costs before committing to it.

In [ ]:
!python -m src.fetch_data --dry-run


In [ ]:
!python -m src.fetch_data --city dar


## 3. Preprocess

Rasterises the footprints onto the imagery grid, indexes tiles, and measures the
class balance. The printed positive fraction is the number the whole experiment
turns on — it is measured here, never assumed, and `weighted_bce` derives its
class weight from it.

In [ ]:
!python -m src.preprocess --city dar


In [ ]:
import json
stats = json.load(open("data/processed/dar/stats.json"))
print("splits          :", stats["splits"])
print("tiles per split :", stats["tiles_per_split"])
print("building pixels :", round(stats["building_pixel_fraction_overall"], 4))
print("empty tiles     :", round(stats["empty_tile_fraction"], 3))
print("validity rule   :", {s: v for s, v in list(stats["scenes"].items())[:1]})


## 4. Train

One arm per call, or `--all` for the full set. Each arm writes
`results/<arm>/` with weights, history, metrics and a georeferenced prediction
raster for the held-out scene.

Start with the control alone to confirm the GPU path works and to see how long
one arm takes, then launch the rest.

In [ ]:
!python -m src.train --arm bce


In [ ]:
!python -m src.train --all


## 5. Pixel-based comparators

CPU-bound and much faster. These measure how much of the task needs no spatial
context at all.

In [ ]:
!python -m src.baseline_pixel --all


## 6. Results table and figures

In [ ]:
!python -m src.figures


In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("results/results_table.md").read()))
for f in ("pr_curves.png", "threshold_tradeoff.png"):
    display(Image(f"figures/{f}"))
import glob
for f in sorted(glob.glob("figures/qualitative_*.png")):
    display(Image(f))


## 7. Save the outputs

Copy `results/` and `figures/` out of the ephemeral Colab disk before the
session ends. The results table and figures are what the README embeds.

In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")

DEST = "/content/gdrive/MyDrive/dar-building-segmentation"
!mkdir -p "$DEST"
!cp -r results figures "$DEST"/
!du -sh "$DEST"
print("copied to", DEST)
